# **SCAPES: simplified tutorial**

Train a SCAPES model with the default settings and generate sounds from it. There are three steps: dataprep, training and generation.

The detailed notebook, `tutorial.ipynb`, has the same steps one by one, with the options explained.

## 1. Setup

Run the first code cell first. On Colab it mounts your Google Drive, clones the SCAPES repository and installs it with the steps of the README. Locally it uses the repository this notebook is in, so install the environment first (README, Installation) and open the notebook from inside the repository.

In [ ]:
# Setup. On Colab: clone SCAPES (with its submodule), mount Drive and install it with the README steps.
# Locally: use the repository this notebook is in (the folder above it that holds SCAPES/pipeline.py).
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"  # branch cloned on Colab

if IN_COLAB:
    from google.colab import drive

    # clone once: the folder /content/SCAPES is kept if the cell runs again
    if not os.path.isdir("/content/SCAPES"):
        subprocess.run(["git", "clone", "--recurse-submodules", "--branch", BRANCH,
                        "https://github.com/cordutie/SCAPES.git", "/content/SCAPES"], check=True)
    os.chdir("/content/SCAPES")
    drive.mount("/content/drive")

    # the install steps of the README (install:env), without the virtual environment
    python_m = [sys.executable, "-m"]
    subprocess.run(python_m + ["pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run(python_m + ["pip", "install", "-r", "requirements.txt",
                               "--extra-index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    subprocess.run(python_m + ["pip", "install", "--no-deps", "msclap==1.3.4",
                               "magenta-rt @ git+https://github.com/magenta/magenta-realtime@694a545e4ba0b88bf1150137b129582166d3e07f"],
                   check=True)
    subprocess.run(python_m + ["pip", "install", "music2latent==0.1.8"], check=True)
    subprocess.run(python_m + ["pip", "install", "-e", "./external/audio_codec_wrappers"], check=True)
    subprocess.run([sys.executable, "external/audio_codec_wrappers/scripts/fetch_weights.py", "--codec", "spectrostream"],
                   check=True)
    REPO = os.getcwd()
else:
    # walk up from the working directory to the repository root
    REPO = os.getcwd()
    while not os.path.exists(os.path.join(REPO, "SCAPES", "pipeline.py")):
        parent = os.path.dirname(REPO)
        if parent == REPO:
            raise FileNotFoundError("Open this notebook from inside the SCAPES repository.")
        REPO = parent

# the repository must be importable as SCAPES and quickstart
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("SCAPES repository:", REPO)

## 2. Parameters

Edit the placeholders in the next cell, or set the environment variables `SCAPES_<NAME>` to run the notebook without editing it.

- The **wav folder** holds your WAV files. They are linked, not copied.
- The **config file** is `config_simplified.gin`: copy it from `SCAPES/config/` and edit it. It sets the codec, the conditioning rate, the conditioning memory, the model size and the cherry-picking CSV. `docs/generation_modes.html` lists the rates each codec offers.
- The **build folder** is where SCAPES writes: `dataset/` (the dataprep output) and `models/<size>_mem<ms>/` (one model per size and memory, for example `large_mem500`).

Dataprep is reused when the config and the wav files are the same as before. A changed codec, rate or wav file is refused: use a different build folder.

In [ ]:
import os

from SCAPES.data.config_loader import load_training_config
from SCAPES.pipeline import model_folder

# Paths. Edit the placeholders, or set the environment variable SCAPES_<NAME> instead.
WAV_PATH = os.environ.get("SCAPES_WAV_PATH", "path/to/your/wavs")          # folder with your WAV files (linked, not copied)
BUILD_PATH = os.environ.get("SCAPES_BUILD_PATH", "path/to/build_folder")   # dataset/ and models/ are written here
CONFIG_PATH = os.environ.get("SCAPES_CONFIG_PATH", "path/to/config.gin")   # a copy of SCAPES/config/config_simplified.gin
SEMANTIC_FLAG = os.environ.get("SCAPES_SEMANTIC_FLAG", "gentle_rain")      # a flag of cherry_picking.csv
SEED = int(os.environ.get("SCAPES_SEED", 0))

# Read from the config file: the model folder (build_path/models/<size>_mem<ms>) and the generation settings
TRAINING = load_training_config(CONFIG_PATH)
MODEL_PATH = model_folder(BUILD_PATH, CONFIG_PATH)
NFE = TRAINING.nfe              # ODE steps (inference.nfe, 16 by default)
CFG_SCALE = TRAINING.cfg_scale  # classifier-free guidance (inference.cfg_scale, 1.0 = off)
print("Model folder:", MODEL_PATH)

## Timing check (about one minute on a GPU)

How fast this machine makes sound with the model size and the generation settings of the config file. The model has random weights (no training, no checkpoint): it makes 10 seconds of sound ten times and prints the average.

In [ ]:
# Timing check: random weights, ten runs of 10 s of sound, with the ODE steps and guidance of the config file.
import time

import torch

from SCAPES.data.config_loader import load_dataprep_config
from SCAPES.inference.FlowInference import FlowInference
from SCAPES.pipeline import units_for_seconds

dataprep = load_dataprep_config(CONFIG_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"
geometry = {"conditioning_rate": dataprep.conditioning_rate,
            "conditioning_memory_seconds": dataprep.conditioning_memory_seconds}
engine = FlowInference.from_config(dataprep.codec_name, size=TRAINING.size, geometry=geometry, device=device)
units = units_for_seconds(engine.geom, 10)
random_sound = lambda: [torch.randn(1024) for _ in range(units)]  # one random CLAP-like vector per unit

engine.generate_soundscape(random_sound(), nfe=TRAINING.nfe, cfg_scale=TRAINING.cfg_scale)  # warm-up: not timed
wall = seconds = 0.0
for _ in range(10):
    start = time.perf_counter()
    audio = engine.generate_soundscape(random_sound(), nfe=TRAINING.nfe, cfg_scale=TRAINING.cfg_scale)
    if device == "cuda":
        torch.cuda.synchronize()
    wall += time.perf_counter() - start
    seconds += audio.shape[-1] / engine.sr
per_second = wall / seconds
gpu = torch.cuda.get_device_name(0) if device == "cuda" else "no GPU (CPU)"
print(f"{gpu}, {dataprep.codec_name}, model '{TRAINING.size}': {per_second:.2f} s to make 1 s of sound. Real time: {'yes' if per_second < 1.0 else 'no'}.")

## 3. Dataprep

Encodes every WAV file of the wav folder with the codec and computes the semantic annotations (CLAP) that the model is conditioned on. The result goes to `BUILD_PATH/dataset/`. If it is already there for the same config and wavs, it is reused and nothing runs.

In [ ]:
from quickstart.wrappers import data_preparation

data_preparation(WAV_PATH, BUILD_PATH, CONFIG_PATH)

In [ ]:
# Free the memory of the step before, without a session reset
import gc

import torch

gc.collect()
torch.cuda.empty_cache()

## 4. Training

Trains the model and saves it in `MODEL_PATH` (`models/<size>_mem<ms>` in the build folder). Training runs until the validation loss stops improving, at most `training.max_epochs` (400). It also writes the semantics folder from the cherry-picking CSV, which the generation step needs. If the dataset is too short for a validation split, the last checkpoint is used. A model folder that already exists is refused: change the size or the memory in the config, or use another build folder.

In [ ]:
from quickstart.wrappers import training_loop

training_loop(WAV_PATH, BUILD_PATH, CONFIG_PATH)

In [ ]:
# Free the memory of the step before, without a session reset
import gc

import torch

gc.collect()
torch.cuda.empty_cache()

## 5. Generation

Generates sound from the semantic vectors of one flag of `cherry_picking.csv` (`SEMANTIC_FLAG`). The output has one unit of audio per stored vector of that flag, so its length follows the segment in the CSV. The file is written to `MODEL_PATH/generated/generation.wav`.

In [ ]:
from pathlib import Path

import torch
from IPython.display import Audio, display

from SCAPES.pipeline import run_generation

# one CLAP vector per row of the flag's file; each row is one unit of audio
prompts = list(torch.load(Path(MODEL_PATH) / "semantics" / f"{SEMANTIC_FLAG}.pt"))
wav = run_generation(model_dir=MODEL_PATH, out_path=Path(MODEL_PATH) / "generated", prompts=prompts,
                     seed=SEED, nfe=NFE, cfg_scale=CFG_SCALE)
display(Audio(str(wav)))